# FreshGuard — Notebook 4 / 5: Train EfficientNetV2‑S Classifier (24 classes)

**Purpose.** Train a 24‑class fine‑grained classifier (`<produce>_<freshness>`) on the deduplicated full split. Used as the **fallback** path when the YOLO26s detector returns zero boxes for an out‑of‑distribution image.

**Why EfficientNetV2‑S.** The PRD is local‑only — no edge / mobile constraint — so we pick for accuracy. 2026 benchmarks show EfficientNetV2‑S wins on macro F1 in the small‑mid model range; MobileNetV3 only wins when latency dominates. ~22 M params, 224 px input, fits comfortably on a Kaggle T4.

**Imbalance handling — three layers stacked.** With a 41:1 class ratio, no single technique is enough.
1. `WeightedRandomSampler` on the train DataLoader using `sample_weight` from the manifest.
2. `CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)` as second line of defense.
3. **Selection metric is macro F1, not top‑1 accuracy** — top‑1 hides minority‑class failure under heavy imbalance.

**Inputs:**
- `ulnnproject/food-freshness-dataset` — raw images
- `<your-handle>/freshguard-manifest` — `manifest.parquet` + `class_weights.json` from notebook 1

**Outputs (publish as `<your-handle>/freshguard-efficientnetv2s`):**
- `efficientnetv2s_food_freshness.pt` — checkpoint with `class_names`, `image_size`, `architecture` baked in (the local strict loader refuses anything else)
- `classifier_metrics.json` — overall + per‑class precision/recall/F1
- `confusion_matrix.png`

**Expected runtime.** ≈4–8 hours on a Kaggle T4 ×2 for 30 epochs.

In [ ]:
!pip install --quiet "timm>=1.0.0" "scikit-learn>=1.3" pyarrow

In [ ]:
from __future__ import annotations

import copy
import json
import math
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from PIL import Image, ImageFile
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
)
from timm.data import create_transform
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

ImageFile.LOAD_TRUNCATED_IMAGES = False

# ---------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------
MANIFEST_PATH = Path("/kaggle/input/datasets/elsmmany/freshguard-manifest/manifest.parquet")
CLASS_WEIGHTS_PATH = Path("/kaggle/input/datasets/elsmmany/freshguard-manifest/class_weights.json")
OUTPUT_DIR = Path("/kaggle/working/classifier_run")
PUBLISH_DIR = Path("/kaggle/working/freshguard-efficientnetv2s")

ARCHITECTURE = "tf_efficientnetv2_s.in21k_ft_in1k"
IMAGE_SIZE = 224
BASE_BATCH_SIZE = 64
_n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
BATCH_SIZE = BASE_BATCH_SIZE * max(1, _n_gpus)
print(f"Detected {_n_gpus} GPU(s). Effective batch size: {BATCH_SIZE}")
NUM_WORKERS = 4

SMOKE_TEST = True
FULL_EPOCHS = 30
WARMUP_EPOCHS = 2
LR = 3e-4
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
MIXUP_ALPHA = 0.2
EMA_DECAY = 0.9999
GRAD_CLIP_NORM = 1.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PUBLISH_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type != "cuda":
    print("WARNING: no GPU detected. Training will be very slow on CPU.")
torch.manual_seed(0)
np.random.seed(0)

In [ ]:
PRODUCE_TYPES = (
    "apple", "banana", "bellpepper", "bitter_gourd", "carrot", "cucumber",
    "mango", "okra", "orange", "potato", "strawberry", "tomato",
)
FRESHNESS_LEVELS = ("fresh", "rotten")
COMBINED_CLASSES = tuple(f"{t}_{f}" for t in PRODUCE_TYPES for f in FRESHNESS_LEVELS)
CLASS_TO_INDEX = {c: i for i, c in enumerate(COMBINED_CLASSES)}
NUM_CLASSES = len(COMBINED_CLASSES)
assert NUM_CLASSES == 24

manifest = pd.read_parquet(MANIFEST_PATH)
OLD_PREFIX = "/kaggle/input/food-freshness-dataset/"
NEW_PREFIX = "/kaggle/input/datasets/ulnnproject/food-freshness-dataset/"
n_translated = int(manifest["path"].str.startswith(OLD_PREFIX).sum())
if n_translated:
    manifest["path"] = manifest["path"].str.replace(OLD_PREFIX, NEW_PREFIX, regex=False)
    print(f"Translated {n_translated:,} legacy paths.")
weights_payload = json.loads(CLASS_WEIGHTS_PATH.read_text())
assert tuple(weights_payload["classes"]) == COMBINED_CLASSES, "class order mismatch"
class_weight_tensor = torch.tensor(weights_payload["weights"], dtype=torch.float32)

if SMOKE_TEST:
    sampled = (
        manifest.groupby("combined_label", group_keys=False)
        .apply(lambda g: g.sample(n=min(40, len(g)), random_state=0))
    )
    manifest = sampled

train_df = manifest[manifest.split == "train"].reset_index(drop=True)
val_df = manifest[manifest.split == "val"].reset_index(drop=True)
test_df = manifest[manifest.split == "test"].reset_index(drop=True)
print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

## Datasets and transforms

`timm.data.create_transform` gives us train transforms with RandAugment + random erasing tuned to the EfficientNetV2 input pipeline. Validation uses simple resize + center crop + ImageNet normalization.

In [ ]:
train_transform = create_transform(
    input_size=IMAGE_SIZE,
    is_training=True,
    auto_augment="rand-m9-mstd0.5",
    interpolation="bicubic",
    re_prob=0.25,
    color_jitter=0.4,
)
eval_transform = create_transform(
    input_size=IMAGE_SIZE,
    is_training=False,
    interpolation="bicubic",
    crop_pct=0.95,
)

class FreshnessDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self) -> int:
        return len(self.df)
    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        image = Image.open(row.path).convert("RGB")
        image = self.transform(image)
        target = CLASS_TO_INDEX[row.combined_label]
        return image, target

train_ds = FreshnessDataset(train_df, train_transform)
val_ds = FreshnessDataset(val_df, eval_transform)
test_ds = FreshnessDataset(test_df, eval_transform)

train_sampler = WeightedRandomSampler(
    weights=train_df.sample_weight.astype(float).values,
    num_samples=len(train_df),
    replacement=True,
)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=train_sampler,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True, persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0,
)
print(f"Batches — train: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)}")

## Model, loss, optimizer, scheduler, EMA

EMA (exponential moving average of weights) is standard practice for vision classifiers in 2026 — gives a meaningful boost in macro F1 with no inference cost (you save the EMA weights and discard the live ones). We save EMA at the end.

In [ ]:
class ModelEma:
    def __init__(self, model: nn.Module, decay: float):
        self.decay = decay
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        for ema_p, p in zip(self.module.parameters(), model.parameters()):
            ema_p.copy_(ema_p * self.decay + p.detach() * (1.0 - self.decay))
        for ema_b, b in zip(self.module.buffers(), model.buffers()):
            ema_b.copy_(b)

model = timm.create_model(ARCHITECTURE, pretrained=True, num_classes=NUM_CLASSES).to(device)
ema = ModelEma(model.module if isinstance(model, torch.nn.DataParallel) else model, EMA_DECAY)

loss_fn = nn.CrossEntropyLoss(weight=class_weight_tensor.to(device), label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

epochs = 2 if SMOKE_TEST else FULL_EPOCHS
warmup = min(WARMUP_EPOCHS, max(1, epochs // 5))

def lr_at(epoch: int) -> float:
    if epoch < warmup:
        return LR * (epoch + 1) / warmup
    progress = (epoch - warmup) / max(1, epochs - warmup)
    return LR * 0.5 * (1.0 + math.cos(math.pi * progress))

scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

def mixup(images: torch.Tensor, targets: torch.Tensor, alpha: float):
    if alpha <= 0:
        return images, targets, targets, 1.0
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(images.size(0), device=images.device)
    return images * lam + images[perm] * (1 - lam), targets, targets[perm], lam

print(f"Model: {ARCHITECTURE} | classes: {NUM_CLASSES} | epochs: {epochs} | warmup: {warmup}")

## Training loop

In [ ]:
@torch.no_grad()
def evaluate(loader: DataLoader, eval_model: nn.Module) -> tuple[float, np.ndarray, np.ndarray]:
    eval_model.eval()
    all_targets, all_preds = [], []
    for images, targets in loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        logits = eval_model(images)
        all_targets.append(targets.cpu().numpy())
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
    targets_arr = np.concatenate(all_targets)
    preds_arr = np.concatenate(all_preds)
    return float(f1_score(targets_arr, preds_arr, average="macro")), targets_arr, preds_arr

best_macro_f1 = -1.0
best_ema_state = None
history = []

for epoch in range(epochs):
    model.train()
    for g in optimizer.param_groups:
        g["lr"] = lr_at(epoch)
    epoch_loss = 0.0
    n_batches = 0
    t0 = time.time()
    for images, targets in train_loader:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        images, t1, t2, lam = mixup(images, targets, MIXUP_ALPHA)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
            logits = model(images)
            loss = lam * loss_fn(logits, t1) + (1 - lam) * loss_fn(logits, t2)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        ema.update(model.module if isinstance(model, torch.nn.DataParallel) else model)
        epoch_loss += float(loss.item())
        n_batches += 1
    train_loss = epoch_loss / max(1, n_batches)
    val_f1, _, _ = evaluate(val_loader, ema.module)
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_macro_f1": val_f1, "lr": lr_at(epoch)})
    elapsed = time.time() - t0
    print(f"epoch {epoch+1:>3}/{epochs}  loss={train_loss:.4f}  val_macroF1={val_f1:.4f}  lr={lr_at(epoch):.2e}  ({elapsed:.0f}s)")
    if val_f1 > best_macro_f1:
        best_macro_f1 = val_f1
        best_ema_state = {k: v.detach().cpu().clone() for k, v in ema.module.state_dict().items()}
        torch.save(
            {
                "model_state_dict": best_ema_state,
                "class_names": list(COMBINED_CLASSES),
                "image_size": IMAGE_SIZE,
                "architecture": ARCHITECTURE,
                "val_macro_f1": best_macro_f1,
                "epoch": epoch + 1,
            },
            OUTPUT_DIR / "best.pt",
        )

print(f"\nBest val macro F1: {best_macro_f1:.4f}")

## Test‑set evaluation + per‑class report

Macro F1 is the headline. Per‑class F1, precision and recall surface where the model fails — typically the smallest classes (Bitter Gourd has only ~684 raw examples).

In [ ]:
ema.module.load_state_dict(best_ema_state)
test_macro_f1, test_targets, test_preds = evaluate(test_loader, ema.module)
report = classification_report(
    test_targets, test_preds,
    labels=list(range(NUM_CLASSES)),
    target_names=list(COMBINED_CLASSES),
    output_dict=True, zero_division=0,
)
cm = confusion_matrix(test_targets, test_preds, labels=list(range(NUM_CLASSES)))

metrics_payload = {
    "architecture": ARCHITECTURE,
    "image_size": IMAGE_SIZE,
    "class_names": list(COMBINED_CLASSES),
    "split": "test",
    "macro_f1": test_macro_f1,
    "top1_accuracy": float((test_preds == test_targets).mean()),
    "per_class": {c: report[c] for c in COMBINED_CLASSES},
    "history": history,
    "hyperparameters": {
        "epochs": epochs, "warmup": warmup, "batch_size": BATCH_SIZE, "lr": LR,
        "weight_decay": WEIGHT_DECAY, "label_smoothing": LABEL_SMOOTHING,
        "mixup_alpha": MIXUP_ALPHA, "ema_decay": EMA_DECAY,
    },
}
(OUTPUT_DIR / "classifier_metrics.json").write_text(json.dumps(metrics_payload, indent=2))

print(f"Test macro F1: {test_macro_f1:.4f}")
print(f"Test top-1:    {metrics_payload['top1_accuracy']:.4f}")
print("\nPer-class F1 (sorted ascending — worst first):")
per_class_f1 = sorted(
    ((c, report[c]["f1-score"]) for c in COMBINED_CLASSES), key=lambda kv: kv[1]
)
for cls, f1 in per_class_f1:
    print(f"  {cls:25s} F1={f1:.3f}  P={report[cls]['precision']:.3f}  R={report[cls]['recall']:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 12))
im = ax.imshow(cm, cmap="viridis")
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels(COMBINED_CLASSES, rotation=90, fontsize=7)
ax.set_yticks(range(NUM_CLASSES))
ax.set_yticklabels(COMBINED_CLASSES, fontsize=7)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Test confusion matrix (macro F1 = {test_macro_f1:.3f})")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
cm_path = OUTPUT_DIR / "confusion_matrix.png"
plt.savefig(cm_path, dpi=120)
plt.show()
print(f"Wrote {cm_path}")

## Stage artifacts for publishing

In [ ]:
import shutil
shutil.copy2(OUTPUT_DIR / "best.pt", PUBLISH_DIR / "efficientnetv2s_food_freshness.pt")
shutil.copy2(OUTPUT_DIR / "classifier_metrics.json", PUBLISH_DIR / "classifier_metrics.json")
shutil.copy2(OUTPUT_DIR / "confusion_matrix.png", PUBLISH_DIR / "confusion_matrix.png")
for p in sorted(PUBLISH_DIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size / 1024 / 1024:.2f} MiB)")

## Publish as a Kaggle Dataset

1. Set `SMOKE_TEST = False` and **Save Version → Save & Run All (Commit)** for the full ~30‑epoch run.
2. From the notebook **Output** tab, **New Dataset** → name it `freshguard-efficientnetv2s`.
3. Notebook 5 attaches this dataset for end‑to‑end evaluation. The local Streamlit app pulls the same `.pt` file via GitHub Releases.

**Verification before publishing:**
- Test macro F1 reported (compare honestly against the previous claimed 0.97 *top‑1*; macro F1 will likely be lower because of the 41:1 imbalance).
- Per‑class F1 table includes every minority class with a real number (not `0.000`).
- Checkpoint file contains `class_names` (the strict loader in the local app refuses anything else).
- Confusion matrix image renders without obvious blocks of off‑diagonal mass.